In [4]:
# ============================================================
# STEP 1a: VERIFY WILLIE WEIGHTS EXIST AND ARE ACCESSIBLE
# ============================================================
# We do NOT download blindly. First, we inspect the repo to see
# what files are actually there — weight format, config, README.

from huggingface_hub import list_repo_files, hf_hub_download

# List all files in the WILLIE weights repository
# This tells us: is it .pt? .safetensors? ONNX? What config files?
repo_id = "QianGroup/willie-weights"

try:
    files = list_repo_files(repo_id)
    print("Files in WILLIE weights repo:")
    for f in files:
        print(f"  - {f}")
except Exception as e:
    print(f"ERROR: Could not access repo. Reason: {e}")
    print("Possible causes: repo is private, renamed, or deleted.")
    # DO NOT proceed if weights are inaccessible.
    # Fallback: check GitHub for inference code that points to correct path.

Files in WILLIE weights repo:
  - .gitattributes
  - README.md
  - assets/WILLIE Workflow.pdf
  - assets/WILLIE Workflow.png
  - base/willie_base_fold0_best.pt
  - base/willie_base_fold1_best.pt
  - base/willie_base_fold2_best.pt
  - base/willie_base_fold3_best.pt
  - base/willie_base_fold4_best.pt
  - decoders/medsam_decoder_best.pt
  - decoders/sam2_decoder_best.pt
  - mini/willie_mini_fold0_best.pt
  - mini/willie_mini_fold1_best.pt
  - mini/willie_mini_fold2_best.pt
  - mini/willie_mini_fold3_best.pt
  - mini/willie_mini_fold4_best.pt
  - xl/willie_xl_fold0_best.pt
  - xl/willie_xl_fold1_best.pt
  - xl/willie_xl_fold2_best.pt
  - xl/willie_xl_fold3_best.pt
  - xl/willie_xl_fold4_best.pt


In [7]:
from huggingface_hub import hf_hub_download, list_repo_files

files = list_repo_files(repo_id)

# Find any license-related files
license_files = [
    f for f in files
    if any(x in f.lower() for x in ["license", "copying"])
]

print("License-related files:", license_files)

# Inspect README license metadata
readme_path = hf_hub_download(
    repo_id=repo_id,
    filename="README.md"
)

with open(readme_path, encoding="utf-8") as f:
    for line in f:
        if "license:" in line.lower():
            print(line.strip())

License-related files: []


license: mit


In [9]:
import requests

url = "https://api.github.com/repos/Qian-Group-HRI/willie/contents"
r = requests.get(url)
r.raise_for_status()

for item in r.json():
    print(item["name"], item["type"])

.gitignore file
01_WILLIE_DataForge.ipynb file
02_WILLIE_DetectionEngine.ipynb file
03_WILLIE_Cls_Baselines.ipynb file
04_WILLIE_Seg_Baselines.ipynb file
05_WILLIE_SAM2_vs_MedSAM.ipynb file
06_WILLIE_MINI.ipynb file
07_WILLIE_BASE.ipynb file
08_WILLIE_XL.ipynb file
09_WILLIE_Scaling_Analysis.ipynb file
10_WILLIE_Cls_Comparison.ipynb file
11_WILLIE_Det_Comparison.ipynb file
Detection_Predictions dir
Figures dir
LICENSE file
Manifests_and_Splits dir
README.md file
SAM2_Generated_Masks dir
Tables dir
ablations dir
data dir
environment.txt file
verify_data.py file


In [10]:
import torch
from huggingface_hub import hf_hub_download

repo_id = "QianGroup/willie-weights"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Download an actual Mini checkpoint
weights_path = hf_hub_download(
    repo_id=repo_id,
    filename="mini/willie_mini_fold0_best.pt"
)

print("Checkpoint downloaded:", weights_path)

# Inspect checkpoint structure safely
checkpoint = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}:")
            for i, (name, tensor) in enumerate(value.items()):
                if i == 10:
                    break
                print(name, getattr(tensor, "shape", type(tensor)))
        elif isinstance(value, torch.Tensor):
            print(key, value.shape)

Device: cuda
Checkpoint downloaded: C:\Users\Lenovo\.cache\huggingface\hub\models--QianGroup--willie-weights\snapshots\2065b069cf9d407cbd1471b0b27cfc2bfc314d64\mini\willie_mini_fold0_best.pt


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weig

In [21]:
checkpoint = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=False
)

In [24]:
import torch
from huggingface_hub import hf_hub_download

repo_id = "QianGroup/willie-weights"
weights_path = hf_hub_download(
    repo_id=repo_id,
    filename="mini/willie_mini_fold0_best.pt"
)

checkpoint = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=False,   # <-- the fix
)

print("Type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))
    sd = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
    print("Total params:", len(sd))
    for i, (k, v) in enumerate(sd.items()):
        if i < 40:
            print(k, tuple(v.shape) if hasattr(v, 'shape') else v)

Type: <class 'dict'>
Top-level keys: ['model_state_dict', 'epoch', 'fold', 'metrics', 'config', 'optimizer_state_dict', 'scheduler_state_dict']
Total params: 380
encoder.backbone.cls_token (1, 1, 384)
encoder.backbone.pos_embed (1, 1370, 384)
encoder.backbone.mask_token (1, 384)
encoder.backbone.patch_embed.proj.weight (384, 3, 14, 14)
encoder.backbone.patch_embed.proj.bias (384,)
encoder.backbone.blocks.0.norm1.weight (384,)
encoder.backbone.blocks.0.norm1.bias (384,)
encoder.backbone.blocks.0.attn.qkv.weight (1152, 384)
encoder.backbone.blocks.0.attn.qkv.bias (1152,)
encoder.backbone.blocks.0.attn.proj.weight (384, 384)
encoder.backbone.blocks.0.attn.proj.bias (384,)
encoder.backbone.blocks.0.ls1.gamma (384,)
encoder.backbone.blocks.0.norm2.weight (384,)
encoder.backbone.blocks.0.norm2.bias (384,)
encoder.backbone.blocks.0.mlp.fc1.weight (1536, 384)
encoder.backbone.blocks.0.mlp.fc1.bias (1536,)
encoder.backbone.blocks.0.mlp.fc2.weight (384, 1536)
encoder.backbone.blocks.0.mlp.fc2.bi

In [25]:
import torch, json
from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(
    repo_id="QianGroup/willie-weights",
    filename="mini/willie_mini_fold0_best.pt"
)
checkpoint = torch.load(weights_path, map_location="cpu", weights_only=False)

# Print the stored config — this may fully define the architecture
print(json.dumps(checkpoint["config"], indent=2, default=str))

# Then print ALL 380 keys (not just 40)
sd = checkpoint["model_state_dict"]
print(f"\nTotal keys: {len(sd)}\n")
for k, v in sd.items():
    print(k, tuple(v.shape) if hasattr(v, 'shape') else v)

{
  "name": "MINI",
  "backbone_name": "dinov2_vits14",
  "backbone_dim": 384,
  "backbone_layers": [
    2,
    5,
    8,
    11
  ],
  "img_size": 518,
  "patch_size": 14,
  "fpn_dim": 256,
  "fpn_levels": 4,
  "wa_csa_layers": 1,
  "wa_csa_heads": 8,
  "wa_csa_dropout": 0.1,
  "num_classes": 5,
  "num_experts": 2,
  "top_k_experts": 2,
  "cls_embed_dim": 128,
  "seg_out_channels": 1,
  "pscse_reduction": 16,
  "det_max_objects": 20,
  "freeze_backbone_epochs": 3
}

Total keys: 380

encoder.backbone.cls_token (1, 1, 384)
encoder.backbone.pos_embed (1, 1370, 384)
encoder.backbone.mask_token (1, 384)
encoder.backbone.patch_embed.proj.weight (384, 3, 14, 14)
encoder.backbone.patch_embed.proj.bias (384,)
encoder.backbone.blocks.0.norm1.weight (384,)
encoder.backbone.blocks.0.norm1.bias (384,)
encoder.backbone.blocks.0.attn.qkv.weight (1152, 384)
encoder.backbone.blocks.0.attn.qkv.bias (1152,)
encoder.backbone.blocks.0.attn.proj.weight (384, 384)
encoder.backbone.blocks.0.attn.proj.bias (

In [26]:
import torch
from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(
    repo_id="QianGroup/willie-weights",
    filename="mini/willie_mini_fold0_best.pt"
)

checkpoint = torch.load(weights_path, map_location="cpu", weights_only=False)
sd = checkpoint["model_state_dict"]

# Print ALL keys
for k, v in sd.items():
    print(k, tuple(v.shape) if hasattr(v, 'shape') else v)

encoder.backbone.cls_token (1, 1, 384)
encoder.backbone.pos_embed (1, 1370, 384)
encoder.backbone.mask_token (1, 384)
encoder.backbone.patch_embed.proj.weight (384, 3, 14, 14)
encoder.backbone.patch_embed.proj.bias (384,)
encoder.backbone.blocks.0.norm1.weight (384,)
encoder.backbone.blocks.0.norm1.bias (384,)
encoder.backbone.blocks.0.attn.qkv.weight (1152, 384)
encoder.backbone.blocks.0.attn.qkv.bias (1152,)
encoder.backbone.blocks.0.attn.proj.weight (384, 384)
encoder.backbone.blocks.0.attn.proj.bias (384,)
encoder.backbone.blocks.0.ls1.gamma (384,)
encoder.backbone.blocks.0.norm2.weight (384,)
encoder.backbone.blocks.0.norm2.bias (384,)
encoder.backbone.blocks.0.mlp.fc1.weight (1536, 384)
encoder.backbone.blocks.0.mlp.fc1.bias (1536,)
encoder.backbone.blocks.0.mlp.fc2.weight (384, 1536)
encoder.backbone.blocks.0.mlp.fc2.bias (384,)
encoder.backbone.blocks.0.ls2.gamma (384,)
encoder.backbone.blocks.1.norm1.weight (384,)
encoder.backbone.blocks.1.norm1.bias (384,)
encoder.backbone.bl

In [27]:
# After loading sd
from collections import defaultdict

groups = defaultdict(list)
for k in sd.keys():
    # Group by top-level module (before first dot)
    top = k.split(".")[0]
    groups[top].append(k)

for group, keys in groups.items():
    print(f"\n{group}: {len(keys)} keys")
    for k in keys[:5]:
        print(f"  {k}")


encoder: 175 keys
  encoder.backbone.cls_token
  encoder.backbone.pos_embed
  encoder.backbone.mask_token
  encoder.backbone.patch_embed.proj.weight
  encoder.backbone.patch_embed.proj.bias

fpn: 24 keys
  fpn.laterals.0.0.weight
  fpn.laterals.0.1.weight
  fpn.laterals.0.1.bias
  fpn.laterals.1.0.weight
  fpn.laterals.1.1.weight

wa_csa: 66 keys
  wa_csa.layers.0.0.alpha_f
  wa_csa.layers.0.0.alpha_c
  wa_csa.layers.0.0.q_f.weight
  wa_csa.layers.0.0.k_c.weight
  wa_csa.layers.0.0.v_c.weight

cls_decoder: 19 keys
  cls_decoder.scale_attn.0.weight
  cls_decoder.scale_attn.0.bias
  cls_decoder.pre.0.weight
  cls_decoder.pre.0.bias
  cls_decoder.pre.1.weight

seg_decoder: 81 keys
  seg_decoder.films.0.gamma.0.weight
  seg_decoder.films.0.gamma.0.bias
  seg_decoder.films.0.beta.weight
  seg_decoder.films.0.beta.bias
  seg_decoder.films.1.gamma.0.weight

det_decoder: 15 keys
  det_decoder.level_w
  det_decoder.shared.0.weight
  det_decoder.shared.1.weight
  det_decoder.shared.1.bias
  det

In [29]:
!git ls-remote https://github.com/Qian-Group-HRI/Willie.git

221f2e7ef470810422436c12730834c82e9bdc8f	HEAD
221f2e7ef470810422436c12730834c82e9bdc8f	refs/heads/main


In [30]:
class WILLIEModel(nn.Module):
    """
    DINOv2 → FPN → WA-CSA → [CLS(MoE) + SEG(P-scSE+FiLM) + DET]
    WTCS bridge: cls wound_embed → FiLM conditioning on seg decoder.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = DINOv2MultiScale(cfg)
        self.fpn = FeaturePyramidNeck(cfg)
        self.wa_csa = WA_CSA_Stack(cfg)
        self.cls_decoder = ClassificationDecoder(cfg)
        self.seg_decoder = SegmentationDecoder(cfg)
        self.det_decoder = DetectionDecoder(cfg)

In [31]:
def forward(self, x, target_seg_size=None):
    pyr = self.wa_csa(self.fpn(self.encoder(x)))
    logits, embed = self.cls_decoder(pyr)
    seg = self.seg_decoder(pyr, embed, target_seg_size)
    det = self.det_decoder(pyr)
    return {
        "cls_logits": logits,           # [B, 5]
        "wound_embed": embed,           # [B, 128]  ← FiLM conditioning
        "seg_mask": seg,                # [B, 1, 518, 518]
        "det_objectness": det["objectness"],  # [B, 1, 37, 37]
        "det_bbox": det["bbox"],        # [B, 4, 37, 37]
        "det_cls": det["det_cls"],      # [B, 5, 37, 37]
    }

In [32]:
def forward(self, x):
    self.features.clear()
    if x.shape[-1] != self.cfg.img_size:
        x = F.interpolate(x, size=self.cfg.img_size, mode="bilinear", align_corners=False)
    self.backbone(x)

In [34]:
import os
import subprocess
import nbformat
from pathlib import Path

# --- Clone the repo if not present ---
REPO_DIR = "Willie"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/Qian-Group-HRI/Willie.git"],
        check=True
    )
    print("Cloned repo.")
else:
    print("Repo already present.")

# --- Pin to the commit we verified ---
subprocess.run(
    ["git", "-C", REPO_DIR, "checkout", "221f2e7ef470810422436c12730834c82e9bdc8f"],
    check=False  # don't fail if already on it
)

# --- Find the MINI notebook ---
nb_path = None
for p in Path(REPO_DIR).rglob("*.ipynb"):
    if "mini" in p.name.lower():
        nb_path = str(p)
        break

if nb_path is None:
    # Fallback: list all notebooks so we can see what's there
    print("No *mini*.ipynb found. All notebooks in repo:")
    for p in Path(REPO_DIR).rglob("*.ipynb"):
        print(f"  {p}")
    raise SystemExit("Specify the notebook path manually.")

print(f"Using notebook: {nb_path}")

# --- Extract all code cells into one .py file ---
nb = nbformat.read(nb_path, as_version=4)

lines = []
lines.append("# Auto-extracted from WILLIE notebook.")
lines.append("# Source: Qian-Group-HRI/Willie @ 221f2e7")
lines.append("# Do not edit by hand — re-run the extractor if needed.\n")

for i, cell in enumerate(nb.cells):
    if cell.cell_type != "code":
        continue
    src = cell.source
    # Skip shell-only lines
    if src.strip().startswith("!"):
        continue
    lines.append(f"# ==== Cell {i} ====")
    lines.append(src)
    lines.append("")

with open("willie_model.py", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("✅ Wrote willie_model.py")

# --- Show what classes and functions it defines ---
import re
with open("willie_model.py") as f:
    content = f.read()

classes = re.findall(r"^class (\w+)", content, re.MULTILINE)
funcs = re.findall(r"^def (\w+)", content, re.MULTILINE)

print(f"\nClasses found ({len(classes)}):")
for c in classes:
    print(f"  class {c}")

print(f"\nTop-level functions found ({len(funcs)}):")
for fn in funcs:
    print(f"  def {fn}()")

Cloned repo.
Using notebook: Willie\06_WILLIE_MINI.ipynb
✅ Wrote willie_model.py


UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 4234: character maps to <undefined>

In [36]:
import ast
import os
import subprocess
import nbformat
from pathlib import Path

# --- Clone if missing ---
REPO_DIR = "Willie"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/Qian-Group-HRI/Willie.git"],
        check=True
    )

# --- Pin to verified commit ---
subprocess.run(
    ["git", "-C", REPO_DIR, "checkout", "221f2e7ef470810422436c12730834c82e9bdc8f"],
    check=False
)

# --- Find MINI notebook ---
nb_path = None
for p in Path(REPO_DIR).rglob("*.ipynb"):
    if "mini" in p.name.lower():
        nb_path = str(p)
        break

assert nb_path, "No *mini*.ipynb found"
print(f"Using notebook: {nb_path}")

nb = nbformat.read(nb_path, as_version=4)

# --- Parse each cell, keep only defs/imports/simple constants ---
imports = []
defs = []
constants = []

def is_simple_constant(node):
    """Module-level NAME = <literal or simple expr>"""
    if not isinstance(node, ast.Assign):
        return False
    # Only allow single-name targets (UPPER or CamelCase or plain)
    if len(node.targets) != 1 or not isinstance(node.targets[0], ast.Name):
        return False
    # Reject calls (they might do I/O)
    for sub in ast.walk(node.value):
        if isinstance(sub, ast.Call):
            return False
    return True

for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    src = cell.source
    if not src.strip():
        continue
    # Skip shell magics
    src = "\n".join(
        line for line in src.splitlines()
        if not line.strip().startswith("!")
    )
    if not src.strip():
        continue
    try:
        tree = ast.parse(src)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            imports.append(ast.unparse(node))
        elif isinstance(node, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)):
            defs.append(ast.unparse(node))
        elif is_simple_constant(node):
            constants.append(ast.unparse(node))

# --- Deduplicate imports ---
seen = set()
uniq_imports = []
for imp in imports:
    if imp not in seen:
        seen.add(imp)
        uniq_imports.append(imp)

# --- Write out ---
header = (
    "# Auto-extracted from WILLIE notebook (classes and functions only).\n"
    "# Source: Qian-Group-HRI/Willie @ 221f2e7\n"
    "# Top-level execution code was stripped by the AST extractor.\n"
)

with open("willie_model.py", "w", encoding="utf-8") as f:
    f.write(header + "\n")
    f.write("\n".join(uniq_imports) + "\n\n")
    if constants:
        f.write("# --- Module-level constants ---\n")
        f.write("\n".join(constants) + "\n\n")
    f.write("\n\n".join(defs))

print("✅ Wrote willie_model.py")

# --- Report what we extracted ---
with open("willie_model.py", encoding="utf-8") as f:
    content = f.read()

import re
classes = re.findall(r"^class (\w+)", content, re.MULTILINE)
funcs = re.findall(r"^def (\w+)", content, re.MULTILINE)

print(f"\nClasses extracted ({len(classes)}):")
for c in classes:
    print(f"  class {c}")

print(f"\nFunctions extracted ({len(funcs)}):")
for fn in funcs:
    print(f"  def {fn}()")

Using notebook: Willie\06_WILLIE_MINI.ipynb
✅ Wrote willie_model.py

Classes extracted (22):
  class ModelConfig
  class DINOv2MultiScale
  class FeaturePyramidNeck
  class WoundAwareCrossScaleAttention
  class WA_CSA_Stack
  class Expert
  class TopKRouter
  class ClassificationDecoder
  class ChannelSE
  class SpatialSE
  class ParallelScSE
  class FiLMConditioner
  class PscSEDecoderStage
  class SegmentationDecoder
  class DetectionDecoder
  class WILLIEModel
  class CheckpointManager
  class WoundMultiTaskDataset
  class DiceLoss
  class MultiTaskLoss
  class MultiTaskLossSafe
  class TTADataset

Functions extracted (26):
  def get_model_config()
  def build_and_verify()
  def create_5fold_splits()
  def get_train_transforms()
  def get_val_transforms()
  def get_tta_transforms()
  def parse_yolo_bbox()
  def mask_to_bboxes()
  def collate_multitask()
  def compute_combined_metric()
  def get_fold_dataloaders()
  def get_test_dataloader()
  def _atomic_save()
  def _load_progress(

In [39]:
%pip install opencv-python einops pillow

   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
    --------------------------------------- 1.0/44.0 MB 5.0 MB/s eta 0:00:09
   -- ------------------------------------- 2.9/44.0 MB 7.0 MB/s eta 0:00:06
   ---- ----------------------------------- 5.0/44.0 MB 8.4 MB/s eta 0:00:05
   ------ --------------------------------- 7.1/44.0 MB 8.7 MB/s eta 0:00:05
   -------- ------------------------------- 8.9/44.0 MB 8.8 MB/s eta 0:00:04
   --------- ------------------------------ 10.2/44.0 MB 8.7 MB/s eta 0:00:04
   ---------- ----------------------------- 11.3/44.0 MB 7.9 MB/s eta 0:00:05
   ----------- ---------------------------- 12.3/44.0 MB 7.6 MB/s eta 0:00:05
   ------------ --------------------------- 14.2/44.0 MB 7.6 MB/s eta 0:00:04
   -------------- ------------------------- 16.0/44.0 MB 7.8 MB/s eta 0:00:04
   --------------- ------------------------ 17.6/44.0 MB 7.8 MB/s eta 0:00:04
   ----

In [46]:
import sys
!{sys.executable} -m pip install albumentations


   ---------------------------------------- 0.0/43.8 MB ? eta -:--:--
   - -------------------------------------- 1.8/43.8 MB 9.1 MB/s eta 0:00:05
   --- ------------------------------------ 4.2/43.8 MB 10.1 MB/s eta 0:00:04
   ----- ---------------------------------- 6.3/43.8 MB 10.4 MB/s eta 0:00:04
   ------- -------------------------------- 8.7/43.8 MB 10.7 MB/s eta 0:00:04
   ---------- ----------------------------- 11.0/43.8 MB 10.6 MB/s eta 0:00:04
   ----------- ---------------------------- 12.8/43.8 MB 10.5 MB/s eta 0:00:03
   ------------- -------------------------- 14.9/43.8 MB 10.3 MB/s eta 0:00:03
   --------------- ------------------------ 16.8/43.8 MB 10.3 MB/s eta 0:00:03
   ----------------- ---------------------- 18.9/43.8 MB 10.2 MB/s eta 0:00:03
   ------------------ --------------------- 20.7/43.8 MB 10.1 MB/s eta 0:00:03
   -------------------- ------------------- 22.8/43.8 MB 10.1 MB/s eta 0:00:03
   ---------------------- ----------------- 24.6/43.8 MB 10.1 MB/s

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\Lenovo\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [49]:
import torch
from huggingface_hub import hf_hub_download
from willie_model import ModelConfig, WILLIEModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

weights_path = hf_hub_download(
    repo_id="QianGroup/willie-weights",
    filename="mini/willie_mini_fold0_best.pt"
)

ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)

cfg = ModelConfig(**ckpt["config"])
print("Config:", cfg)

model = WILLIEModel(cfg).to(device)

result = model.load_state_dict(ckpt["model_state_dict"], strict=True)
print("\n✅ strict=True passed. All 380 weights loaded, no missing/unexpected.")
print(f"Epoch: {ckpt.get('epoch')}, Fold: {ckpt.get('fold')}")
print(f"Metrics: {ckpt.get('metrics')}")

model.eval()

Device: cuda
Config: ModelConfig(name='MINI', backbone_name='dinov2_vits14', backbone_dim=384, backbone_layers=[2, 5, 8, 11], img_size=518, patch_size=14, fpn_dim=256, fpn_levels=4, wa_csa_layers=1, wa_csa_heads=8, wa_csa_dropout=0.1, num_classes=5, num_experts=2, top_k_experts=2, cls_embed_dim=128, seg_out_channels=1, pscse_reduction=16, det_max_objects=20, freeze_backbone_epochs=3)


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to C:\Users\Lenovo/.cache\torch\hub\main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to C:\Users\Lenovo/.cache\torch\hub\checkpoints\dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:13<00:00, 6.33MB/s]



✅ strict=True passed. All 380 weights loaded, no missing/unexpected.
Epoch: 9, Fold: 0
Metrics: {'epoch': 9, 'best_combined': np.float64(81.89247212969887), 'cls_acc': 84.72222222222221, 'cls_f1': 84.75371071059315, 'seg_dice': np.float64(79.17700061334831), 'seg_n': 202, 'det_ap50': np.float64(81.66391497735334), 'det_n': 202, 'loss': -0.018201999905230252, 'combined_equal': np.float64(81.85437927097462), 'combined_weighted': np.float64(81.89247212969887), 'combined_cls_seg': np.float64(81.94961141778526), 'min_task': np.float64(79.17700061334831)}


WILLIEModel(
  (encoder): DINOv2MultiScale(
    (backbone): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-11): 12 x NestedTensorBlock(
          (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
          (attn): MemEffAttention(
            (qkv): Linear(in_features=384, out_features=1152, bias=True)
            (proj): Linear(in_features=384, out_features=384, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=384, out_features=1536, bias=True)
            (act): GELU(approximate='none')
            (fc2): Linear(in_features=1536, out_features=384, bias=True)
            (drop): Dropout(p=0.0, inpla

In [51]:
import torch
from huggingface_hub import hf_hub_download
from willie_model import ModelConfig, WILLIEModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

weights_path = hf_hub_download(
    repo_id="QianGroup/willie-weights",
    filename="mini/willie_mini_fold0_best.pt"
)

# Load the checkpoint dict (authors use weights_only=False)
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)

# Build config from the config embedded in the checkpoint
cfg = ModelConfig(**ckpt["config"])

# Instantiate the real model
model = WILLIEModel(cfg).to(device)

# THE TEST: strict=True raises if any key is missing or unexpected
try:
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    print("\n✅ strict=True PASSED — model is correct, all 380 weights loaded.")
    print(f"Epoch:   {ckpt.get('epoch')}")
    print(f"Fold:    {ckpt.get('fold')}")
    print(f"Metrics: {ckpt.get('metrics')}")
    model.eval()
    print("Model in eval mode. Ready for inference.")
except RuntimeError as e:
    print("\n❌ strict=True FAILED:")
    print(str(e))

Device: cuda


Using cache found in C:\Users\Lenovo/.cache\torch\hub\facebookresearch_dinov2_main



✅ strict=True PASSED — model is correct, all 380 weights loaded.
Epoch:   9
Fold:    0
Metrics: {'epoch': 9, 'best_combined': np.float64(81.89247212969887), 'cls_acc': 84.72222222222221, 'cls_f1': 84.75371071059315, 'seg_dice': np.float64(79.17700061334831), 'seg_n': 202, 'det_ap50': np.float64(81.66391497735334), 'det_n': 202, 'loss': -0.018201999905230252, 'combined_equal': np.float64(81.85437927097462), 'combined_weighted': np.float64(81.89247212969887), 'combined_cls_seg': np.float64(81.94961141778526), 'min_task': np.float64(79.17700061334831)}
Model in eval mode. Ready for inference.


In [56]:
# ============================================================
# INFERENCE: one image through the verified WILLIEModel
# ============================================================
import torch
import torch.nn.functional as F
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

# --- Input size: model interpolates to 518 internally ---
# We feed 518 directly to avoid double interpolation.
INPUT_SIZE = 518

# --- Standard ImageNet normalization (DINOv2 convention) ---
preprocess = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def run_willie(image_path, model, device):
    """
    Run WILLIEModel on one image.
    Returns the 6 raw outputs + original RGB image for visualization.
    """
    # OpenCV loads BGR — convert to RGB for PIL/torchvision
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Preprocess → [1, 3, 518, 518] on device
    tensor = preprocess(Image.fromarray(img_rgb)).unsqueeze(0).to(device)

    # Forward pass — no gradient (inference only)
    with torch.no_grad():
        out = model(tensor)  # returns dict with 6 keys

    return out, img_rgb


# --- Provide your test image path here ---
from pathlib import Path
_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMAGE_PATH = str(_root / "sample_wound.jpg")   # project root
assert Path(IMAGE_PATH).exists(), f"image not found: {IMAGE_PATH}"

out, img_rgb = run_willie(IMAGE_PATH, model, device)

# --- Inspect the 6 outputs ---
print("Output keys and shapes:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: shape={tuple(v.shape)}, "
              f"min={v.min().item():.3f}, max={v.max().item():.3f}")

# --- Classification: softmax over 5 classes ---
# Class names from WILLIE README: diabetic, pressure, surgical, venous, no_wound
CLASSES = ["diabetic", "pressure", "surgical", "venous", "no_wound"]
probs = F.softmax(out["cls_logits"], dim=1)[0].cpu().numpy()
pred = int(probs.argmax())

print(f"\nClassification: {CLASSES[pred]}  ({probs[pred]:.3f})")
for name, p in zip(CLASSES, probs):
    print(f"  {name:10s}: {p:.4f}")

# --- Segmentation: sigmoid → binary mask at 0.5 ---
seg_logits = out["seg_mask"]               # [1, 1, 518, 518]
seg_prob = torch.sigmoid(seg_logits)[0, 0].cpu().numpy()
seg_bin = (seg_prob > 0.5).astype(np.uint8)
wound_pct = seg_bin.sum() / seg_bin.size * 100

print(f"\nSegmentation: wound covers {wound_pct:.2f}% of image")

# --- Detection: objectness map — where is the peak? ---
obj = out["det_objectness"][0, 0].cpu().numpy()   # [37, 37]
peak = np.unravel_index(obj.argmax(), obj.shape)
print(f"Detection: peak objectness {obj.max():.3f} at grid {peak} (of 37x37)")

# --- Visualize: 4 panels ---
h, w = img_rgb.shape[:2]

# Resize probability map back to original image size for overlay
seg_prob_resized = cv2.resize(seg_prob, (w, h), interpolation=cv2.INTER_LINEAR)
seg_bin_resized = cv2.resize(seg_bin, (w, h), interpolation=cv2.INTER_NEAREST)

overlay = img_rgb.copy()
overlay[seg_bin_resized == 1] = [255, 0, 0]
blended = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img_rgb); ax[0].set_title("Original"); ax[0].axis("off")
ax[1].imshow(seg_prob_resized, cmap="jet")
ax[1].set_title("Seg probability"); ax[1].axis("off")
ax[2].imshow(blended)
ax[2].set_title(f"Mask overlay ({wound_pct:.1f}% area)")
ax[2].axis("off")
plt.tight_layout(); plt.show()

FileNotFoundError: sample_wound.jpg